In [2]:
import numpy as np
import pandas as pd

SEED = 40

main = pd.read_stata("../data/raw/coding_suseti_pmt.dta", convert_categoricals=False)
labels = pd.read_stata("../data/raw/mistargeting_CORRECTED.dta", convert_categoricals=False)

df = main.merge(labels[["hhid_suseti", "poor"]], left_on="hhid", right_on="hhid_suseti")
print("Shape:", df.shape)
print(f"Share poor: {df['poor'].mean():.1%}")

Shape: (5756, 349)
Share poor: 35.3%


In [3]:
print("main:", [c for c in main.columns if "share" in c.lower() or "quota" in c.lower()])
print("labels:", [c for c in labels.columns if "share" in c.lower() or "quota" in c.lower()])

main: ['SHAREAGRICULTURALHH', 'quotaslsb1dec', 'share', 'quota', 'quota_final', 'quota_2']
labels: ['SHAREAGRICULTURALHH', 'quota_final']


In [4]:
quota_cols = ["share", "quota", "quota_final", "quota_2"]

display(df[["hhea"] + quota_cols].head(10))
display(df[quota_cols].describe())

print(df.groupby("hhea")[quota_cols].nunique().max())

df["n_slots"] = (8 * df["share"]).round()
for c in ["quota", "quota_final", "quota_2"]:
    print(f"n_slots == {c}: {(df['n_slots'] == df[c]).mean():.1%}")


,hhea,share,quota,quota_final,quota_2
0,1,0.405405,28.783783,29,15
1,1,0.405405,28.783783,29,15
2,1,0.405405,28.783783,29,15
3,1,0.405405,28.783783,29,15
4,1,0.405405,28.783783,29,15
5,1,0.405405,28.783783,29,15
6,1,0.405405,28.783783,29,15
7,1,0.405405,28.783783,29,15
8,1,0.405405,28.783783,29,15
9,2,0.482759,29.448277,29,15


,share,quota,quota_final,quota_2
count,5756.000000,5756.000000,5756.000000,5756.000000
mean,0.321068,16.705946,16.719076,8.617269
std,0.176160,13.824646,13.827451,6.925893
min,0.012500,0.833333,1.000000,1.000000
25%,0.183333,7.809524,8.000000,4.000000
50%,0.295455,13.282051,13.000000,7.000000
75%,0.430556,22.000000,22.000000,11.000000
max,0.854545,168.068436,168.000000,84.000000


share          1
quota          1
quota_final    1
quota_2        1
dtype: int64
n_slots == quota: 0.0%
n_slots == quota_final: 0.0%
n_slots == quota_2: 2.0%


/var/folders/c4/tgg8_grs3mb8cwzhfy7m2yp80000gn/T/ipykernel_58543/1992747218.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["n_slots"] = (8 * df["share"]).round()


In [5]:
print(df.groupby("hhea").size().value_counts())

9    636
8      4
Name: count, dtype: int64


In [6]:
def select_by_quota(data, score, higher_is_poorer):
    n_slots = (8 * data["share"]).round()
    rank = data.groupby("hhea")[score].rank(ascending=not higher_is_poorer, method="first")
    return (rank <= n_slots).astype(int)

In [7]:
df = df.copy()
df["selected_gov"] = select_by_quota(df, "pcexppred", higher_is_poorer=False)

display(df[df["hhea"] == 1][["hhea", "share", "pcexppred", "selected_gov"]])

print(df.groupby("hhea")["selected_gov"].sum().head())

,hhea,share,pcexppred,selected_gov
0,1,0.405405,257589.06250,1
1,1,0.405405,258868.56250,0
2,1,0.405405,412232.50000,0
3,1,0.405405,386270.84375,0
4,1,0.405405,415046.15625,0
5,1,0.405405,487587.93750,0
6,1,0.405405,339553.15625,0
7,1,0.405405,240098.96875,1
8,1,0.405405,180716.75000,1


hhea
1    3
2    4
3    4
4    3
5    3
Name: selected_gov, dtype: int64


In [10]:
def recall_at_quota(data, selected):
    poor_selected = ((selected == 1) & (data["poor"] == 1)).sum()
    all_poor = (data["poor"] == 1).sum()
    return poor_selected / all_poor

print(f"Recall government formula: {recall_at_quota(df, df['selected_gov']):.1%}")

Recall government formula: 49.4%


In [12]:
# Random Baseline

rng = np.random.default_rng(SEED)

random_recalls = []
for i in range(100):
    df["random_score"] = rng.random(len(df))
    selected = select_by_quota(df, "random_score", higher_is_poorer=True)
    random_recalls.append(recall_at_quota(df, selected))
    
recall_random = np.mean(random_recalls)
print(f"Recall random selection: {recall_random:.1%}")


Recall random selection: 31.7%


In [13]:
# Perfect ranking (the ceiling)

selected = select_by_quota(df, "LOGCONSUMPTION", higher_is_poorer=False)
recall_perfect = recall_at_quota(df, selected)
print(f"Recall perfect ranking: {recall_perfect:.1%}")

Recall perfect ranking: 64.7%


In [14]:
recall_gov = recall_at_quota(df, df["selected_gov"])

results = pd.DataFrame({
    "model": ["Random", "Government formula", "Perfect ranking"],
    "recall": [recall_random, recall_gov, recall_perfect]
})
display(results)

improvement = (recall_gov - recall_random) / (recall_perfect - recall_random)
print(f"The government formula reaches {improvement:.0%} of the possible improvement.")


,model,recall
0,Random,0.317377
1,Government formula,0.494083
2,Perfect ranking,0.647436


The government formula reaches 54% of the possible improvement.
